In [21]:
import sys
import json
import yaml
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")

print("Python:", sys.executable)
print("프로젝트:", ROOT)

PLAN_ROOT = (
    ROOT.parent
    / "datasets/hemorrhage/v1/model_ready/nnunet"
    / "nnUNet_preprocessed/Dataset001_ICHBinary"
)

# nnU-Net reader 설정 수정
for name in [
    "nnUNetResEncUNetMPlans.json",
    "ICHPlansResEncMV1.json",
    "ICHPlansResEncMAttentionV1.json",
]:
    path = PLAN_ROOT / name
    data = json.loads(path.read_text(encoding="utf-8"))
    data["image_reader_writer"] = "NibabelIO"
    path.write_text(
        json.dumps(
            data,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("수정:", path.name)

# SegResNet worker를 0으로 설정
for experiment in ["H-S10", "H-S20"]:
    path = (
        ROOT
        / f"configs/experiments/hemorrhage/{experiment}.yaml"
    )
    config = yaml.safe_load(
        path.read_text(encoding="utf-8")
    )
    config["data"]["num_workers"] = 0
    path.write_text(
        yaml.safe_dump(
            config,
            allow_unicode=True,
            sort_keys=False,
        ),
        encoding="utf-8",
    )
    print("수정:", path.name)

Python: /home/stu03/workspace/model/.venv/bin/python
프로젝트: /home/stu03/workspace/model
수정: nnUNetResEncUNetMPlans.json
수정: ICHPlansResEncMV1.json
수정: ICHPlansResEncMAttentionV1.json
수정: H-S10.yaml
수정: H-S20.yaml


In [22]:
import json
import yaml
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")

PLAN_ROOT = (
    ROOT.parent
    / "datasets/hemorrhage/v1/model_ready/nnunet"
    / "nnUNet_preprocessed/Dataset001_ICHBinary"
)

print("=== nnU-Net reader 확인 ===")

for name in [
    "nnUNetResEncUNetMPlans.json",
    "ICHPlansResEncMV1.json",
    "ICHPlansResEncMAttentionV1.json",
]:
    path = PLAN_ROOT / name
    data = json.loads(path.read_text(encoding="utf-8"))
    reader = data.get("image_reader_writer")

    print(f"{name}: {reader}")
    assert reader == "NibabelIO"

print()
print("=== SegResNet worker 확인 ===")

for experiment in ["H-S10", "H-S20"]:
    path = (
        ROOT
        / f"configs/experiments/hemorrhage/{experiment}.yaml"
    )
    config = yaml.safe_load(
        path.read_text(encoding="utf-8")
    )
    workers = config["data"]["num_workers"]

    print(f"{experiment}: num_workers={workers}")
    assert workers == 0

print()
print("모든 수정 사항이 정상적으로 적용돼 있습니다.")

=== nnU-Net reader 확인 ===
nnUNetResEncUNetMPlans.json: NibabelIO
ICHPlansResEncMV1.json: NibabelIO
ICHPlansResEncMAttentionV1.json: NibabelIO

=== SegResNet worker 확인 ===
H-S10: num_workers=0
H-S20: num_workers=0

모든 수정 사항이 정상적으로 적용돼 있습니다.


In [23]:
# 실행 전 점검

import subprocess

result = subprocess.run(
    [
        sys.executable,
        "scripts/run_hemorrhage_variants.py",
        "--dry-run",
        "--only",
        "H-N20",
        "H-S10",
        "H-S20",
        "--gpu",
        "0",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

data_root=/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet
segresnet_root=/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/segresnet
manifest=/home/stu03/workspace/datasets/hemorrhage/v1/metadata/common/06_preprocessing_manifest.csv
H-N20: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.nnunet.runner --config /home/stu03/workspace/model/configs/experiments/hemorrhage/H-N20.yaml --data-root /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet --results-root /home/stu03/workspace/model/runs/nnunet
H-S10: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.models.hemorrhage.segresnet.train --config /home/stu03/workspace/model/configs/experiments/hemorrhage/H-S10.yaml
H-S20: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.models.hemorrhage.segresnet.train --config /home/stu03/workspace/model/configs/experiments/hemorrhage/H-S20.yaml



In [24]:
# H-N10 best-composite validation만 재실행

import os
import sys
import torch
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")

DATA_ROOT = (
    ROOT.parent
    / "datasets/hemorrhage/v1/model_ready/nnunet"
)

RESULTS_ROOT = ROOT / "runs/nnunet"

source_path = str(ROOT / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)

# SHM 사용 최소화
os.environ["nnUNet_n_proc_DA"] = "0"
os.environ["nnUNet_def_n_proc"] = "1"

from brainon_ai.nnunet.config import set_config_environment
from brainon_ai.nnunet.runner import (
    TRAINER_NAME,
    configure_paths,
)
from nnunetv2.run.run_training import get_trainer_from_args

config_path = (
    ROOT
    / "configs/experiments/hemorrhage/H-N10.yaml"
)

set_config_environment(config_path)
configure_paths(DATA_ROOT, RESULTS_ROOT)

trainer = get_trainer_from_args(
    "1",
    "3d_fullres",
    0,
    TRAINER_NAME,
    "ICHPlansResEncMV1",
    False,
    device=torch.device("cuda"),
)

checkpoint = (
    RESULTS_ROOT
    / "Dataset001_ICHBinary"
    / "nnUNetTrainerICHBaseline__ICHPlansResEncMV1__3d_fullres"
    / "fold_0"
    / "checkpoint_best_composite.pth"
)

print("Checkpoint:", checkpoint)
print("존재 여부:", checkpoint.exists())

trainer.load_checkpoint(checkpoint)
trainer.perform_actual_validation(False)

print("H-N10 best-composite validation 완료")

/home/stu03/workspace/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Trainer 'nnUNetTrainerICHBaseline' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerICHBaseline in folder /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
Found class nnUNetTrainerICHBaseline in configurable_trainer
Using trainer 'nnUNetTrainerICHBaseline' from: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
##############################################################

In [ ]:
# import os
# import sys
# import time
# import signal
# import subprocess
# from pathlib import Path

# ROOT = Path("/home/stu03/workspace/model")
# RUN_DIR = ROOT / "runs/hemorrhage-variants"
# PID_FILE = RUN_DIR / "batch.pid"
# LOG_FILE = RUN_DIR / "launcher.log"

# RUN_DIR.mkdir(parents=True, exist_ok=True)

# # 기존 배치가 실행 중이면 종료
# if PID_FILE.exists():
#     old_pid = int(PID_FILE.read_text().strip())

#     try:
#         os.kill(old_pid, 0)
#         print(f"기존 배치 종료 중: PID={old_pid}")
#         os.killpg(old_pid, signal.SIGTERM)
#         time.sleep(3)
#     except ProcessLookupError:
#         print("기존 PID는 이미 종료된 상태입니다.")

# # 실행 환경 설정
# environment = os.environ.copy()
# environment["nnUNet_n_proc_DA"] = "0"
# environment["nnUNet_def_n_proc"] = "2"

# # 새 배치 실행
# with LOG_FILE.open("a", encoding="utf-8") as log:
#     process = subprocess.Popen(
#         [
#             sys.executable,
#             "scripts/run_hemorrhage_variants.py",
#             "--gpu",
#             "0",
#         ],
#         cwd=ROOT,
#         env=environment,
#         stdout=log,
#         stderr=subprocess.STDOUT,
#         start_new_session=True,
#     )

# PID_FILE.write_text(str(process.pid))

# print(f"새 배치 실행 PID: {process.pid}")
# print("nnU-Net worker: 2")
# print("실행 순서: H-N10 → H-N20 → H-S10 → H-S20")
# print(f"로그: {LOG_FILE}")

기존 배치 종료 중: PID=73839
새 배치 실행 PID: 76386
nnU-Net worker: 2
실행 순서: H-N10 → H-N20 → H-S10 → H-S20
로그: /home/stu03/workspace/model/runs/hemorrhage-variants/launcher.log


In [25]:
# H-N20, H-S10, H-S20 백그라운드 실행

import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")
RUN_DIR = ROOT / "runs/hemorrhage-variants"
PID_FILE = RUN_DIR / "batch.pid"
LOG_FILE = RUN_DIR / "launcher_remaining.log"

RUN_DIR.mkdir(parents=True, exist_ok=True)

environment = os.environ.copy()
environment["nnUNet_n_proc_DA"] = "0"
environment["nnUNet_def_n_proc"] = "1"

with LOG_FILE.open("a", encoding="utf-8") as log:
    process = subprocess.Popen(
        [
            sys.executable,
            "scripts/run_hemorrhage_variants.py",
            "--only",
            "H-N20",
            "H-S10",
            "H-S20",
            "--gpu",
            "0",
        ],
        cwd=ROOT,
        env=environment,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

PID_FILE.write_text(str(process.pid))

print("실행 PID:", process.pid)
print("nnU-Net worker: 0")
print("실행 순서: H-N20 → H-S10 → H-S20")
print("로그:", LOG_FILE)

실행 PID: 123210
nnU-Net worker: 0
실행 순서: H-N20 → H-S10 → H-S20
로그: /home/stu03/workspace/model/runs/hemorrhage-variants/launcher_remaining.log


In [26]:
# 실시간 진행률 표시
BATCH_STATUS = RUN_DIR / "batch_status.json"

PROGRESS_FILES = {
    "H-N10": (
        ROOT / "runs/nnunet/Dataset001_ICHBinary/"
        "nnUNetTrainerICHBaseline__ICHPlansResEncMV1__3d_fullres/"
        "fold_0/progress.json"
    ),
    "H-N20": (
        ROOT / "runs/nnunet/Dataset001_ICHBinary/"
        "nnUNetTrainerICHBaseline__ICHPlansResEncMAttentionV1__3d_fullres/"
        "fold_0/progress.json"
    ),
    "H-S10": ROOT / "runs/segresnet/H-S10/progress.json",
    "H-S20": ROOT / "runs/segresnet/H-S20/progress.json",
}


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def format_time(seconds):
    if seconds is None:
        return "-"

    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


while True:
    clear_output(wait=True)

    batch = read_json(BATCH_STATUS)
    batch_items = batch.get("experiments", {})
    rows = []

    for experiment, path in PROGRESS_FILES.items():
        progress = read_json(path)
        batch_item = batch_items.get(experiment, {})

        rows.append({
            "실험": experiment,
            "상태": progress.get(
                "status",
                batch_item.get("status", "waiting"),
            ),
            "Epoch": (
                f"{progress.get('current_epoch', 0)}"
                f"/{progress.get('total_epochs', '-')}"
            ),
            "진행률(%)": progress.get("progress_percent", 0),
            "ETA": format_time(progress.get("eta_seconds")),
            "Loss": progress.get("train_loss"),
            "Dice": progress.get(
                "dice",
                progress.get("pseudo_dice"),
            ),
            "Recall": progress.get(
                "recall",
                progress.get("pseudo_recall"),
            ),
            "Best Composite": progress.get("best_composite"),
            "GPU 메모리(GB)": progress.get("gpu_max_memory_gb"),
        })

    print("전체 상태:", batch.get("status", "starting"))
    print("현재 실험:", batch.get("current", "-"))
    print()

    display(pd.DataFrame(rows))

    if batch.get("status") in {"completed", "failed"}:
        print("배치 실행이 종료됐습니다.")
        break

    time.sleep(10)

전체 상태: completed
현재 실험: None



,실험,상태,Epoch,진행률(%),ETA,Loss,Dice,Recall,Best Composite,GPU 메모리(GB)
0,H-N10,early_stopped,229/1000,22.9,05:12:11,NaN,NaN,NaN,0.842841,NaN
1,H-N20,early_stopped,229/1000,22.9,05:25:59,NaN,NaN,NaN,0.862736,NaN
2,H-S10,early_stopped,240/300,80.0,02:22:13,0.579230,0.515465,0.564953,0.551983,2.193
3,H-S20,completed,300/300,100.0,00:00:00,0.554016,0.526892,0.559076,0.549307,2.071


배치 실행이 종료됐습니다.


In [8]:
import os
import signal
import time
from pathlib import Path

RUN_DIR = Path(
    "/home/stu03/workspace/model/runs/hemorrhage-variants"
)

pid = int((RUN_DIR / "batch.pid").read_text().strip())
os.killpg(pid, signal.SIGTERM)
time.sleep(5)

print("종료 요청 완료:", pid)

종료 요청 완료: 73839


In [27]:
for experiment, path in PROGRESS_FILES.items():
    progress = read_json(path)

    elapsed = progress.get("session_elapsed_seconds")
    mean_epoch = progress.get("mean_epoch_seconds")

    print(
        experiment,
        "총 시간:",
        format_time(elapsed),
        "평균 epoch:",
        (
            f"{mean_epoch:.2f}초"
            if mean_epoch is not None
            else "-"
        ),
    )

H-N10 총 시간: 01:35:21 평균 epoch: 24.30초
H-N20 총 시간: 01:39:25 평균 epoch: 25.37초
H-S10 총 시간: 09:28:54 평균 epoch: 142.23초
H-S20 총 시간: 12:58:13 평균 epoch: 155.64초


In [ ]:
# 현재 로그 확인
batch = read_json(BATCH_STATUS)
current = batch.get("current")

if current:
    log_file = RUN_DIR / "logs" / f"{current}.log"

    if log_file.exists():
        lines = log_file.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        print("\n".join(lines[-50:]))
    else:
        print("로그 파일 생성 대기 중")
else:
    print("현재 실행 중인 실험이 없습니다.")